# TabDPT Classifier Artifact Inference — DIMER tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/tutorials/tabdpt_classifier_artifact_inference_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Layer6%2FTabDPT-ffcc4d?style=flat)](https://huggingface.co/Layer6/TabDPT)

**Profile:** `ARTIFACT-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification `1.0`  
**Repository code revision exercised:** `99eaf21986bd74a1336f3aa2ea32407eeea884ef`

This notebook consumes an **externally supplied** `tabdpt-dimer-context-v3` artifact (`artifact.json` + `training_context.parquet`), validates its identity and support-context digest before reconstruction, restores the fitted preprocessing state, conditions the pinned TabDPT foundation model on the saved support context, then scores **externally supplied new data** without refitting preprocessing.

`load_artifact()` reconstructs serving state; it does not gradient-train or fine-tune TabDPT. The upstream checkpoint remains immutable and is acquired separately from `Layer6/TabDPT` at the repository-pinned revision with SHA-256 verification.

**By the end of this notebook you will be able to:**
- validate an externally supplied DIMER v3 artifact before reconstruction;
- inspect model identity, revision, artifact format, class order, and context digest;
- reconstruct the serving object with fitted preprocessing restored from artifact state;
- validate and score new unlabelled CSV records;
- export predictions and provenance in machine-readable formats.

**This notebook does not create its own artifact.** Produce one in a separate execution of the E2E notebook or DIMER runtime, then supply it here.


## Prerequisites and trust boundary

- **Environment:** fresh Google Colab or compatible Jupyter runtime; Python 3.11–3.13.
- **Accelerator:** GPU recommended; CPU supported but slower. `use_flash=False` is forced for Tesla T4 portability.
- **External inputs required:** `artifact.json`, `training_context.parquet`, and a new inference CSV.
- **Privacy:** supplied files stay in the notebook runtime unless explicitly exported elsewhere. Do not upload restricted data to a hosted runtime without authorization.
- **Archive policy:** this notebook intentionally accepts individual files, **not ZIP/TAR archives**. Archive extraction security requirements are therefore not applicable to this workflow.

**Trust boundary:** a matching manifest and SHA-256 prove internal byte consistency, not sender authenticity. JSON and Parquet are treated as data inputs; the base checkpoint is separately anchored to the repository-pinned immutable revision and SHA-256. If a future artifact format includes pickle, Python objects, or other code-capable serialization, do not load it without an explicit trusted-producer policy.


In [ ]:
import sys
if "torch" in sys.modules:
    raise RuntimeError(
        "Start from a fresh runtime: PyTorch is already imported, and the pinned tutorial install "
        "must complete before core ML packages are loaded."
    )

REPO_REVISION = "99eaf21986bd74a1336f3aa2ea32407eeea884ef"
REPO_DIR = "/content/tabdpt-classifier-pipeline"

!rm -rf "$REPO_DIR"
!git clone -q https://github.com/kurtvalcorza/tabdpt-classifier-pipeline.git "$REPO_DIR"
!git -C "$REPO_DIR" checkout -q "$REPO_REVISION"
!python -m pip install -q -r "$REPO_DIR/tutorials/requirements-colab.txt"
!python -m pip install -q --no-deps "$REPO_DIR"


## 1. Verify runtime and expected model identity

The repository fixes the supported base model to TabDPT v1.2 / TabDPT-Turbo. The next cell prints the effective runtime and expected immutable model identity. This is the identity an uploaded artifact must declare.


In [ ]:
import importlib.metadata as mdlib
import platform
import sys
import torch

from tabdpt_classifier_pipeline import (
    TABDPT_HF_REPO,
    TABDPT_HF_REVISION,
    TABDPT_UPSTREAM_CODE_COMMIT,
    TABDPT_WEIGHT_FILENAME,
    TABDPT_WEIGHT_SHA256,
    TabDPTClassificationPipeline,
)
from tabdpt_classifier_pipeline.pipeline import resolve_tabdpt_weights

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
for package in ["tabdpt", "torch", "numpy", "pandas", "scikit-learn", "huggingface-hub", "pyarrow"]:
    print(f"{package}:", mdlib.version(package))
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Expected model:", TABDPT_HF_REPO)
print("Expected revision:", TABDPT_HF_REVISION)
print("Expected weight SHA-256:", TABDPT_WEIGHT_SHA256)
print("Tutorial attention mode: use_flash=False")


## 2. Supply and validate the external artifact

Upload exactly these two files from a **previous, separate** producing execution:

- `artifact.json`
- `training_context.parquet`

The notebook validates the manifest as data before model reconstruction. It rejects the wrong artifact format/task, unexpected model identity, inconsistent target/drop/class metadata, non-local support-context paths, missing files, oversized support context, and SHA-256 mismatch. It also checks that fitted preprocessing metadata is present.

No archive is accepted, so there is no extraction step and no opportunity for archive path traversal or symlink extraction.


In [ ]:
import hashlib
import json
from pathlib import Path, PurePosixPath

ARTIFACT_DIR = Path("/content/external-tabdpt-artifact")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError(
        "This Colab path expects upload(). In local Jupyter, place artifact.json and "
        f"training_context.parquet under {ARTIFACT_DIR} before running this cell."
    ) from exc

uploaded = files.upload()
allowed_names = {"artifact.json", "training_context.parquet"}
uploaded_names = {Path(name).name for name in uploaded}
if uploaded_names != allowed_names:
    raise ValueError(f"Upload exactly {sorted(allowed_names)}; received {sorted(uploaded_names)}")
for name, payload in uploaded.items():
    (ARTIFACT_DIR / Path(name).name).write_bytes(payload)

manifest_path = ARTIFACT_DIR / "artifact.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if not isinstance(manifest, dict):
    raise ValueError("artifact.json must contain a JSON object")
if manifest.get("format") != "tabdpt-dimer-context-v3":
    raise ValueError(f"Unsupported artifact format: {manifest.get('format')!r}")
if manifest.get("taskType") != "tabular_classification":
    raise ValueError(f"Artifact taskType must be 'tabular_classification', got {manifest.get('taskType')!r}")

base_model = manifest.get("baseModel")
if not isinstance(base_model, dict):
    raise ValueError("Artifact is missing baseModel metadata")
expected_model = {
    "repo": TABDPT_HF_REPO,
    "revision": TABDPT_HF_REVISION,
    "filename": TABDPT_WEIGHT_FILENAME,
    "sha256": TABDPT_WEIGHT_SHA256,
    "upstreamCodeCommit": TABDPT_UPSTREAM_CODE_COMMIT,
}
mismatches = {
    key: (base_model.get(key), expected)
    for key, expected in expected_model.items()
    if base_model.get(key) != expected
}
if mismatches:
    raise ValueError(f"Artifact base-model identity mismatch: {mismatches}")

preprocessing = manifest.get("preprocessing")
if not isinstance(preprocessing, dict) or preprocessing.get("schemaVersion") != 1:
    raise ValueError("Artifact is missing supported preprocessing schemaVersion=1 state")
encoder = preprocessing.get("encoder")
if not isinstance(encoder, dict) or not encoder.get("featureColumns"):
    raise ValueError("Artifact preprocessing is missing fitted featureColumns")
if manifest.get("targetColumn") != preprocessing.get("targetColumn"):
    raise ValueError("Artifact targetColumn disagrees with fitted preprocessing state")
if list(manifest.get("dropColumns", [])) != list(preprocessing.get("dropColumns", [])):
    raise ValueError("Artifact dropColumns disagree with fitted preprocessing state")
if list(manifest.get("classNames", [])) != list(preprocessing.get("classLabels", [])):
    raise ValueError("Artifact classNames disagree with fitted preprocessing classLabels")

training_context = manifest.get("trainingContext")
if not isinstance(training_context, dict):
    raise ValueError("Artifact is missing trainingContext metadata")
context_rel = training_context.get("path")
if not isinstance(context_rel, str) or "\\" in context_rel:
    raise ValueError("trainingContext.path must be a portable POSIX relative path")
context_posix = PurePosixPath(context_rel)
if context_posix.is_absolute() or ".." in context_posix.parts or len(context_posix.parts) != 1:
    raise ValueError("trainingContext.path must name one file in the artifact directory")
context_path = ARTIFACT_DIR / context_posix.name
if context_path.name != "training_context.parquet":
    raise ValueError(f"Expected training_context.parquet, got {context_path.name!r}")
if not context_path.is_file():
    raise FileNotFoundError(f"Missing support context: {context_path}")
MAX_CONTEXT_BYTES = 512 * 1024**2
if context_path.stat().st_size > MAX_CONTEXT_BYTES:
    raise ValueError(f"Support context exceeds {MAX_CONTEXT_BYTES} bytes")

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

expected_context_sha = training_context.get("sha256")
if not isinstance(expected_context_sha, str) or len(expected_context_sha) != 64:
    raise ValueError("trainingContext.sha256 must be a SHA-256 hex digest")
actual_context_sha = sha256_file(context_path)
if actual_context_sha != expected_context_sha:
    raise RuntimeError(
        f"Support-context digest mismatch: expected {expected_context_sha}, got {actual_context_sha}"
    )

print("Artifact format:", manifest["format"])
print("Task:", manifest["taskType"])
print("Model:", base_model["repo"])
print("Model revision:", base_model["revision"])
print("Weight SHA-256:", base_model["sha256"])
print("Support-context SHA-256:", actual_context_sha)
print("Class order:", manifest.get("classNames"))
print("Fitted feature count:", len(encoder["featureColumns"]))
print("Artifact pre-reconstruction validation: PASS")


## 3. Reconstruct the serving state

`resolve_tabdpt_weights()` acquires/verifies the repository-pinned `.safetensors` checkpoint. `load_artifact()` then restores the saved preprocessing state and conditions a new TabDPT estimator on `training_context.parquet`.

Although the upstream estimator internally uses a method named `fit()` to register support context, this reconstruction performs **no gradient training** and does **not refit the repository's feature encoder from the uploaded inference data**.


In [ ]:
weights = resolve_tabdpt_weights()
pipe = TabDPTClassificationPipeline.load_artifact(
    manifest_path,
    model_weight_path=weights,
    compile_model=False,
    use_flash=False,
)
print("Serving state reconstructed: PASS")
print("Restored target column:", pipe.target_column)
print("Restored class order:", pipe.class_labels_)
print("Restored feature columns:", pipe.feature_encoder.feature_columns)
print("Restored preprocessing; no inference-data refit occurred")


## 4. Supply and validate genuinely new input

Upload exactly one unlabelled CSV. Before upload, the expected serving schema is:

- fitted feature columns: shown by the previous cell;
- optional configured `dropColumns`: may be present and are removed by the serving contract;
- target column: **must not** be present for inference;
- duplicate headers: rejected before pandas can rename them;
- all required fitted features: must be present;
- other unexpected columns: rejected.

Configured drop columns are preserved in the exported prediction table as identifiers when present. Otherwise the notebook emits a stable `row_id` derived from the input row index.


In [ ]:
import csv
from collections import Counter
from pathlib import Path
import pandas as pd

INPUT_DIR = Path("/content/tabdpt-new-input")
INPUT_DIR.mkdir(parents=True, exist_ok=True)

uploaded_input = files.upload()
if len(uploaded_input) != 1:
    raise ValueError("Upload exactly one new inference CSV")
input_name, input_payload = next(iter(uploaded_input.items()))
if Path(input_name).suffix.lower() != ".csv":
    raise ValueError("New inference input must be a CSV file")
input_path = INPUT_DIR / Path(input_name).name
input_path.write_bytes(input_payload)

with input_path.open("r", encoding="utf-8-sig", newline="") as handle:
    reader = csv.reader(handle)
    try:
        header = next(reader)
    except StopIteration as exc:
        raise ValueError("Inference CSV is empty") from exc
duplicates = sorted(name for name, count in Counter(header).items() if count > 1)
if duplicates:
    raise ValueError(f"Inference CSV contains duplicate column names: {duplicates}")

new_data = pd.read_csv(input_path)
if pipe.target_column in new_data.columns:
    raise ValueError(
        f"Inference CSV contains target column {pipe.target_column!r}; remove labels before scoring"
    )

configured_drop = [col for col in pipe.drop_columns_ if col in new_data.columns]
required = list(pipe.feature_encoder.feature_columns)
effective_columns = [col for col in new_data.columns if col not in configured_drop]
missing = sorted(set(required) - set(effective_columns))
extra = sorted(set(effective_columns) - set(required))
if missing or extra:
    raise ValueError(f"Inference feature schema mismatch; missing={missing}, extra={extra}")

print("New input shape:", new_data.shape)
print("Preserved identifier/drop columns:", configured_drop)
print("Schema validation: PASS")


## 5. Predict and export machine-readable results

The default classification decision is **argmax over class scores**. `predict_proba()` supplies one score per class in the artifact's explicit class order. These outputs are not claimed to be calibrated probabilities; deployment-specific calibration/thresholding requires labelled domain data.

Predictions are exported as CSV, and provenance is exported as JSON.


In [ ]:
import importlib.metadata as mdlib
import json
import pandas as pd

INFERENCE = {
    "n_ensembles": 2,
    "context_size": 512,
    "batch_size": 512,
    "temperature": 1.0,
    "seed": int(manifest.get("preprocessing", {}).get("seed", 42)),
}
scores = pipe.predict_proba(new_data, **INFERENCE)
predictions = pipe.predict(new_data, **INFERENCE)

if configured_drop:
    output = new_data[configured_drop].copy()
else:
    output = pd.DataFrame({"row_id": new_data.index.astype(str)})
output["prediction"] = predictions.astype(str).to_numpy()
for class_name in pipe.class_labels_:
    output[f"score_{class_name}"] = scores[class_name].to_numpy()

OUTPUT_DIR = Path("/content/tabdpt-artifact-inference-output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
predictions_path = OUTPUT_DIR / "predictions.csv"
provenance_path = OUTPUT_DIR / "provenance.json"
output.to_csv(predictions_path, index=False)

provenance = {
    "repository": "kurtvalcorza/tabdpt-classifier-pipeline",
    "repositoryRevision": REPO_REVISION,
    "notebookProfile": "ARTIFACT-INFERENCE",
    "notebookSpecVersion": "1.0",
    "artifact": {
        "format": manifest["format"],
        "contextSha256": actual_context_sha,
        "baseModel": base_model,
        "classOrder": list(pipe.class_labels_),
    },
    "inference": INFERENCE,
    "runtime": {
        package: mdlib.version(package)
        for package in ["tabdpt", "torch", "numpy", "pandas", "scikit-learn", "huggingface-hub", "pyarrow"]
    },
    "useFlash": False,
}
provenance_path.write_text(json.dumps(provenance, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print(output.head())
print("Predictions:", predictions_path)
print("Provenance:", provenance_path)
print("Decision rule: argmax")
print("Calibration status: not established by this tutorial")


## Interpretation and troubleshooting

A successful run proves that, in this pinned runtime, an artifact supplied from **outside this notebook execution** can be checked for DIMER v3 identity and support-context integrity, reconstructed with the repository's saved preprocessing state, combined with the immutable checksum-verified TabDPT v1.2 checkpoint, and used to score separately supplied new records without refitting preprocessing.

It does **not prove** sender authenticity, model calibration, robustness, fairness, domain validity, or production fitness. The support context remains part of the served model state and may contain confidential or licensed training/support data. Internal SHA-256 consistency does not authenticate who supplied the artifact.

Common failures are intentional and actionable:
- **model identity mismatch** — use an artifact produced by the matching repository/model revision;
- **support digest mismatch** — reacquire the artifact; do not bypass the check;
- **schema mismatch** — provide exactly the fitted feature set plus any configured drop/identifier columns;
- **CUDA/FlashAttention issue** — this notebook forces `use_flash=False`; CPU remains a slower fallback;
- **network failure while resolving weights** — retry only when the pinned model host is reachable or provide the verified pinned checkpoint through an authorized local cache.

Before release, record a clean-runtime execution for the exact notebook/PR revision. Static validation alone is not execution evidence.
